# NovaLM-ULTRA - Google Colab Training 🚀
## 1B Model Training on FineWeb Dataset

**Author:** NovaLM Team

Yeh notebook aapko **Google Colab** mein NovaLM-ULTRA 1B model train karne degi.
Sab kuch auto-install ho jayega - bas **Runtime > Run all** click karein!

---
## Step 1: ⚡ Check GPU & Install Dependencies

In [ ]:
# ===== CHECK GPU =====
import torch, sys, os, subprocess, json, shutil
print(f"🐍 Python: {sys.version}")
print(f"🔥 PyTorch: {torch.__version__}")
cuda_avail = torch.cuda.is_available()
if cuda_avail:
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU! CPU mode (slow)")
    print("   Runtime > Change runtime type > T4 GPU select karein")

In [ ]:
# ===== CLONE REPO =====
REPO_URL = "https://github.com/anupbth1/nova_core_pro_V2.git"
FOLDER_NAME = "nova_core_pro_V2"

if not os.path.exists(FOLDER_NAME):
    !git clone --depth 1 {REPO_URL}
    print("✅ Repo cloned!")
else:
    print(f"✅ {FOLDER_NAME} already exists, pulling latest...")
    %cd {FOLDER_NAME}
    !git pull
    %cd ..

%cd {FOLDER_NAME}/NovaLM-ULTRA

In [ ]:
# ===== INSTALL DEPENDENCIES =====
import subprocess
deps = ["torch", "numpy", "tqdm", "datasets", "tokenizers", "transformers", 
        "accelerate", "safetensors", "psutil"]
for d in deps:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", d], capture_output=True)
print("✅ All dependencies installed!")

---
## Step 2: 🏗️ Create Model & Start Training

Neeche diye gaye command se 1B parameter model train hoga **FineWeb dataset** par.

In [ ]:
# ===== 1B MODEL TRAINING COMMAND =====
# Yeh command 1 Billion parameter model train karega FineWeb dataset par
# Aap sirf YEHI CELL chalayein - baaki sab auto hai!

!python scripts/train_ultra_complete.py ^
  --dim 1024 --layers 16 --heads 16 --kv-heads 4 --head-dim 64 --ffn-dim 4096 ^
  --vocab-size 32000 --block-size 512 --weight-share 1 ^
  --dataset "HuggingFaceFW/fineweb" --dataset-subset "sample-10BT" ^
  --epochs 5 --batch-size 8 ^
  --lr 3e-4 --warmup-steps 100 ^
  --mixed-precision fp16 ^
  --save-dir ./models/1B_model

---
## 🔧 Troubleshooting (अगर कोई error aaye toh)

### Error 1: "CUDA out of memory"
**Solution:** Batch size kam karein:
```
!python scripts/train_ultra_complete.py --dim 1024 --layers 16 --heads 16 --kv-heads 4 --vocab-size 32000 --block-size 512 --dataset HuggingFaceFW/fineweb --dataset-subset sample-10BT --epochs 5 --batch-size 2 --mixed-precision fp16
```

### Error 2: Dataset slow download
**Solution:** Streaming mode use karein:
```
!python scripts/train_ultra_complete.py --dim 1024 --layers 16 --heads 16 --vocab-size 32000 --dataset HuggingFaceFW/fineweb --streaming --epochs 5 --batch-size 8 --mixed-precision fp16
```

### Error 3: Google Colab disconnect
**Solution:** Colab mein `Runtime > Run all` ke baad browser tab ko open rakhein.
Ya **Colab Pro** use karein for longer training.

### Error 4: ModuleNotFoundError
**Solution:** Upar wala install cell dobara chalayein ya:
```
!pip install torch numpy tqdm datasets transformers accelerate
```

---
## 📊 1B Model - Parameter Calculation

| Parameter | Value | Calculation |
|-----------|-------|------------|
| dim | 1024 | - |
| layers | 16 | - |
| heads | 16 | - |
| kv-heads | 4 | 4x KV cache savings |
| head-dim | 64 | dim/heads = 1024/16 = 64 |
| ffn-dim | 4096 | dim*4 = 4096 |
| vocab-size | 32000 | ~32k tokens |
| Total Params | ~1B | 16 layers × (attention + FFN) + embedding |
| VRAM needed | ~8GB | batch_size=8, fp16 |

✅ **T4 GPU (15GB VRAM)** par aaram se chalega!

---
## 🎯 Quick Test (1 epoch, TinyStories - INSTANT)

Agar sirf test karna hai toh ye use karein (1 minute):

In [ ]:
# ===== QUICK TEST (1 epoch, ~1 minute) =====
# TinyStories dataset - bohot chhota, instantly download!

!python scripts/train_ultra_complete.py ^
  --dataset roneneldan/TinyStories ^
  --dim 256 --layers 4 --heads 4 --ffn-dim 1024 ^
  --block-size 128 --epochs 1 --batch-size 8

---
## 📁 Saved Models

Training ke baad models yahan save honge:
- `nova_core_pro_V2/NovaLM-ULTRA/models/1B_model/`

Download karne ke liye:
```python
from google.colab import files
!zip -r /content/model_backup.zip /content/nova_core_pro_V2/NovaLM-ULTRA/models/
files.download("/content/model_backup.zip")
```